In [2]:
import pandas as pd
import numpy as np

# Load raw match data
raw_matches = pd.read_csv("atp_matches_2021_2024.csv")

# Filter matches with all required stats
stats_cols = [
    'w_1stWon', 'w_2ndWon', 'w_df', 'w_ace', 'w_svpt', 'w_1stIn',
    'l_1stWon', 'l_2ndWon', 'l_df', 'l_ace', 'l_svpt', 'l_1stIn'
]
filtered_matches = raw_matches.dropna(subset=stats_cols + ['surface', 'winner_name', 'loser_name'])

# Prepare features
feature_rows = []
for _, row in filtered_matches.iterrows():
    match = {
        'player_1': row['winner_name'],
        'player_2': row['loser_name'],
        'surface': row['surface'],
        'round': row['round'],
        'best_of': row['best_of'],
        'tourney_level': row['tourney_level'],
        'p1_age': row['winner_age'],
        'p2_age': row['loser_age'],
        'p1_ht': row['winner_ht'],
        'p2_ht': row['loser_ht'],
        'p1_rank': row['winner_rank'],
        'p2_rank': row['loser_rank'],
        'p1_1stWon%': row['w_1stWon'] / row['w_1stIn'] if row['w_1stIn'] else 0,
        'p2_1stWon%': row['l_1stWon'] / row['l_1stIn'] if row['l_1stIn'] else 0,
        'p1_2ndWon%': row['w_2ndWon'] / (row['w_svpt'] - row['w_1stIn']) if (row['w_svpt'] - row['w_1stIn']) else 0,
        'p2_2ndWon%': row['l_2ndWon'] / (row['l_svpt'] - row['l_1stIn']) if (row['l_svpt'] - row['l_1stIn']) else 0,
        'p1_df': row['w_df'],
        'p2_df': row['l_df'],
        'p1_ace': row['w_ace'],
        'p2_ace': row['l_ace'],
        'outcome': 1
    }
    feature_rows.append(match)

    flipped = match.copy()
    flipped['player_1'], flipped['player_2'] = match['player_2'], match['player_1']
    flipped['p1_age'], flipped['p2_age'] = match['p2_age'], match['p1_age']
    flipped['p1_ht'], flipped['p2_ht'] = match['p2_ht'], match['p1_ht']
    flipped['p1_rank'], flipped['p2_rank'] = match['p2_rank'], match['p1_rank']
    flipped['p1_1stWon%'], flipped['p2_1stWon%'] = match['p2_1stWon%'], match['p1_1stWon%']
    flipped['p1_2ndWon%'], flipped['p2_2ndWon%'] = match['p2_2ndWon%'], match['p1_2ndWon%']
    flipped['p1_df'], flipped['p2_df'] = match['p2_df'], match['p1_df']
    flipped['p1_ace'], flipped['p2_ace'] = match['p2_ace'], match['p1_ace']
    flipped['outcome'] = 0
    feature_rows.append(flipped)

model_c_df = pd.DataFrame(feature_rows)

# Add deltas
model_c_df['delta_1stWon'] = model_c_df['p1_1stWon%'] - model_c_df['p2_1stWon%']
model_c_df['delta_2ndWon'] = model_c_df['p1_2ndWon%'] - model_c_df['p2_2ndWon%']
model_c_df['delta_df'] = model_c_df['p1_df'] - model_c_df['p2_df']
model_c_df['delta_ace'] = model_c_df['p1_ace'] - model_c_df['p2_ace']
model_c_df['delta_age'] = model_c_df['p1_age'] - model_c_df['p2_age']
model_c_df['delta_ht'] = model_c_df['p1_ht'] - model_c_df['p2_ht']
model_c_df['delta_rank'] = model_c_df['p1_rank'] - model_c_df['p2_rank']

# Save
model_c_df.to_csv("model_c_dataset.csv", index=False)
